# Phase 5: OCR & Document Intelligence
## Day 21: TesseractBasics

Date: 2026-04-24

### Learning objectives
- Understand what Tesseract OCR does.
- Use pytesseract safely from Python.
- Learn language packs, PSM modes, and OEM modes.
- Extract text and bounding boxes.
- Build a small OCR result cleanup workflow.

In [ ]:
import json
import re
import shutil
import textwrap
from pprint import pprint

import pandas as pd
import numpy as np

try:
    from PIL import Image, ImageDraw, ImageFont, ImageFilter
    PIL_AVAILABLE = True
except Exception:
    PIL_AVAILABLE = False

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except Exception:
    MATPLOTLIB_AVAILABLE = False

try:
    import pytesseract
    PYTESSERACT_AVAILABLE = True
except Exception:
    pytesseract = None
    PYTESSERACT_AVAILABLE = False

TESSERACT_BINARY_AVAILABLE = shutil.which("tesseract") is not None

def show(title, content):
    print("\n" + "=" * 76)
    print(title)
    print("=" * 76)
    print(textwrap.dedent(str(content)).strip())

print("Setup complete.")
print("PIL available:", PIL_AVAILABLE)
print("pytesseract available:", PYTESSERACT_AVAILABLE)
print("Tesseract binary available:", TESSERACT_BINARY_AVAILABLE)

In [ ]:
sample_receipt_text = '''
BERLIN COFFEE BAR
Date: 2026-04-24
Latte             4.20 EUR
Croissant         3.10 EUR
Bagel             5.50 EUR
Total            12.80 EUR
'''

sample_invoice_text = '''
Invoice ID: INV-1024
Customer: Spring Coffee Push
Channel: Instagram
Spend: 1200 EUR
Clicks: 3420
Conversions: 184
'''

print(sample_receipt_text)
print(sample_invoice_text)

## 1. Create a sample image inline

OCR means reading text from an image.

To keep this notebook self-contained, we create a small receipt image with Python. No external files are needed.

In [ ]:
def create_text_image(text, width=760, height=360, font_size=24):
    if not PIL_AVAILABLE:
        return None

    image = Image.new("RGB", (width, height), color="white")
    draw = ImageDraw.Draw(image)

    try:
        font = ImageFont.truetype("DejaVuSansMono.ttf", font_size)
    except Exception:
        font = ImageFont.load_default()

    draw.multiline_text((30, 30), text.strip(), fill="black", font=font, spacing=10)
    return image

receipt_image = create_text_image(sample_receipt_text)

if receipt_image is not None:
    display(receipt_image)
else:
    print("PIL is not available, so the image cannot be displayed.")

In [ ]:
invoice_image = create_text_image(sample_invoice_text, width=820, height=340)

if invoice_image is not None:
    display(invoice_image)
else:
    print("PIL is not available, so the invoice image cannot be displayed.")

## 2. Basic OCR with pytesseract

`pytesseract` is a Python wrapper. It calls the Tesseract OCR engine installed on your computer.

If Tesseract is not installed, this notebook returns a mock OCR result so the examples still run.

In [ ]:
def mock_ocr(image_name="receipt"):
    if image_name == "invoice":
        return sample_invoice_text.strip()
    return sample_receipt_text.strip()

def run_ocr(image, lang="eng", config="", image_name="receipt"):
    if image is None:
        return mock_ocr(image_name)

    if PYTESSERACT_AVAILABLE and TESSERACT_BINARY_AVAILABLE:
        try:
            return pytesseract.image_to_string(image, lang=lang, config=config)
        except Exception as error:
            print("Real OCR failed. Using mock OCR instead.")
            print("Error:", error)
            return mock_ocr(image_name)

    print("pytesseract or Tesseract is not available. Using mock OCR.")
    return mock_ocr(image_name)

ocr_text = run_ocr(receipt_image, image_name="receipt")
show("OCR text output", ocr_text)

In [ ]:
invoice_ocr_text = run_ocr(invoice_image, image_name="invoice")
show("Invoice OCR output", invoice_ocr_text)

## 3. Language packs

Tesseract uses language packs. English is usually `eng`. German is `deu`. Turkish is `tur`.

You can combine languages with a plus sign, such as `eng+deu`.

In [ ]:
language_examples = {
    "English only": "eng",
    "German only": "deu",
    "Turkish only": "tur",
    "English plus German": "eng+deu",
    "English plus Turkish": "eng+tur"
}

pprint(language_examples)

for label, lang_code in language_examples.items():
    print(f"{label}: lang='{lang_code}'")

In [ ]:
def build_tesseract_command(image_path, lang="eng", psm=6, oem=3):
    return f"tesseract {image_path} stdout -l {lang} --psm {psm} --oem {oem}"

print(build_tesseract_command("receipt.png", lang="eng", psm=6, oem=3))
print(build_tesseract_command("german_note.png", lang="deu", psm=6, oem=3))
print(build_tesseract_command("mixed_text.png", lang="eng+deu", psm=6, oem=3))

## 4. PSM modes

PSM means page segmentation mode. It tells Tesseract what kind of layout to expect.

For clean blocks of text, `--psm 6` is a common starting point. For one line, try `--psm 7`. For one word, try `--psm 8`.

In [ ]:
psm_modes = pd.DataFrame([
    {"psm": 3, "meaning": "Fully automatic page segmentation", "good_for": "Full pages"},
    {"psm": 4, "meaning": "Single column of text", "good_for": "Scanned articles"},
    {"psm": 6, "meaning": "Single uniform block of text", "good_for": "Receipts and simple forms"},
    {"psm": 7, "meaning": "Single text line", "good_for": "One-line labels"},
    {"psm": 8, "meaning": "Single word", "good_for": "Word crops"},
    {"psm": 11, "meaning": "Sparse text", "good_for": "Text scattered around image"},
])

psm_modes

In [ ]:
for psm in [3, 6, 7, 11]:
    config = f"--psm {psm}"
    text = run_ocr(receipt_image, config=config, image_name="receipt")
    print("\nPSM:", psm)
    print(text[:160])

## 5. OEM modes

OEM means OCR engine mode.

In modern Tesseract, `--oem 3` means use the default available engine. This is usually a safe default.

In [ ]:
oem_modes = pd.DataFrame([
    {"oem": 0, "meaning": "Legacy engine only"},
    {"oem": 1, "meaning": "Neural LSTM engine only"},
    {"oem": 2, "meaning": "Legacy plus LSTM"},
    {"oem": 3, "meaning": "Default based on what is available"},
])

oem_modes

In [ ]:
for oem in [1, 3]:
    config = f"--oem {oem} --psm 6"
    text = run_ocr(invoice_image, config=config, image_name="invoice")
    print("\nOEM:", oem)
    print(text[:180])

## 6. Bounding boxes

OCR can return not only text, but also where each word appears.

Bounding boxes are useful for document intelligence because they keep layout information.

In [ ]:
def mock_ocr_data_for_receipt():
    rows = [
        {"text": "BERLIN", "conf": 95, "left": 30, "top": 30, "width": 90, "height": 24},
        {"text": "COFFEE", "conf": 96, "left": 130, "top": 30, "width": 100, "height": 24},
        {"text": "BAR", "conf": 96, "left": 240, "top": 30, "width": 50, "height": 24},
        {"text": "Date:", "conf": 93, "left": 30, "top": 74, "width": 70, "height": 24},
        {"text": "2026-04-24", "conf": 94, "left": 110, "top": 74, "width": 160, "height": 24},
        {"text": "Latte", "conf": 94, "left": 30, "top": 118, "width": 80, "height": 24},
        {"text": "4.20", "conf": 92, "left": 310, "top": 118, "width": 60, "height": 24},
        {"text": "EUR", "conf": 92, "left": 385, "top": 118, "width": 45, "height": 24},
        {"text": "Total", "conf": 96, "left": 30, "top": 250, "width": 80, "height": 24},
        {"text": "12.80", "conf": 91, "left": 300, "top": 250, "width": 75, "height": 24},
        {"text": "EUR", "conf": 91, "left": 390, "top": 250, "width": 45, "height": 24},
    ]
    return pd.DataFrame(rows)

def get_ocr_data(image, lang="eng", config="--psm 6"):
    if image is not None and PYTESSERACT_AVAILABLE and TESSERACT_BINARY_AVAILABLE:
        try:
            data = pytesseract.image_to_data(
                image,
                lang=lang,
                config=config,
                output_type=pytesseract.Output.DATAFRAME
            )
            data = data.dropna(subset=["text"])
            data = data[data["text"].astype(str).str.strip() != ""]
            return data[["text", "conf", "left", "top", "width", "height"]].reset_index(drop=True)
        except Exception as error:
            print("Real bounding-box OCR failed. Using mock data.")
            print("Error:", error)

    print("Using mock OCR bounding boxes.")
    return mock_ocr_data_for_receipt()

ocr_data = get_ocr_data(receipt_image)
ocr_data

In [ ]:
high_conf_words = ocr_data[ocr_data["conf"] >= 90].copy()
high_conf_words

In [ ]:
def draw_boxes(image, data):
    if image is None or not PIL_AVAILABLE:
        print("Image drawing is not available.")
        return None

    boxed = image.copy()
    draw = ImageDraw.Draw(boxed)

    for _, row in data.iterrows():
        left = int(row["left"])
        top = int(row["top"])
        right = left + int(row["width"])
        bottom = top + int(row["height"])
        draw.rectangle([left, top, right, bottom], outline="red", width=2)

    return boxed

boxed_image = draw_boxes(receipt_image, high_conf_words)

if boxed_image is not None:
    display(boxed_image)

## 7. Cleaning OCR text

OCR output can contain extra spaces, broken lines, or small mistakes.

Start with simple cleanup before moving to complex correction.

In [ ]:
def clean_ocr_text(text):
    text = text.replace("\x0c", "")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()

messy_text = "BERLIN   COFFEE BAR\n\nLatte      4.20 EUR\x0c"
cleaned = clean_ocr_text(messy_text)

print("Before:")
print(repr(messy_text))
print("\nAfter:")
print(repr(cleaned))

In [ ]:
clean_receipt_text = clean_ocr_text(ocr_text)
show("Cleaned receipt text", clean_receipt_text)

## 8. Extract fields from OCR text

OCR is often only the first step.

After OCR, you usually extract structured fields from the text.

In [ ]:
def extract_receipt_fields(text):
    clean = clean_ocr_text(text)

    date_match = re.search(r"Date:\s*(\d{4}-\d{2}-\d{2})", clean)
    total_match = re.search(r"Total\s+([0-9]+\.[0-9]{2})\s+EUR", clean)

    items = []
    for line in clean.splitlines():
        match = re.search(r"^([A-Za-z]+)\s+([0-9]+\.[0-9]{2})\s+EUR$", line.strip())
        if match and match.group(1).lower() != "total":
            items.append({
                "item": match.group(1),
                "price_eur": float(match.group(2))
            })

    return {
        "date": date_match.group(1) if date_match else None,
        "total_eur": float(total_match.group(1)) if total_match else None,
        "items": items
    }

receipt_fields = extract_receipt_fields(clean_receipt_text)
pprint(receipt_fields)

In [ ]:
items_total = sum(item["price_eur"] for item in receipt_fields["items"])

print("Items total:", round(items_total, 2))
print("Receipt total:", receipt_fields["total_eur"])
print("Matches:", abs(items_total - receipt_fields["total_eur"]) < 0.01)

## 9. Tesseract install notes

You need two things for real OCR:

1. The Tesseract app installed on your computer.
2. The Python package `pytesseract`.

On macOS, you can usually install Tesseract with Homebrew.

In [ ]:
install_notes = '''
macOS:
brew install tesseract
brew install tesseract-lang
pip install pytesseract pillow

Ubuntu:
sudo apt-get install tesseract-ocr
sudo apt-get install tesseract-ocr-deu tesseract-ocr-tur
pip install pytesseract pillow

Python check:
import pytesseract
pytesseract.image_to_string(image, lang="eng", config="--psm 6 --oem 3")
'''

print(install_notes)

## Tricky bits

OCR problems are usually caused by image quality, layout, wrong language packs, or wrong PSM mode.

Do not expect perfect text from bad images. Preprocessing comes later in Phase 5.

In [ ]:
tricky_cases = pd.DataFrame([
    {
        "problem": "Wrong language",
        "symptom": "Special characters are misread",
        "fix": "Use the right lang code, such as deu or tur"
    },
    {
        "problem": "Wrong PSM",
        "symptom": "Text order looks strange",
        "fix": "Try psm 6 for blocks, psm 7 for one line"
    },
    {
        "problem": "Low contrast",
        "symptom": "Missing or broken words",
        "fix": "Use grayscale and thresholding"
    },
    {
        "problem": "Skewed image",
        "symptom": "Lines are not read well",
        "fix": "Deskew before OCR"
    },
    {
        "problem": "Tiny text",
        "symptom": "Numbers are wrong",
        "fix": "Increase resolution before OCR"
    },
])

tricky_cases

In [ ]:
def explain_ocr_failure(symptom):
    symptom = symptom.lower()
    if "special" in symptom or "character" in symptom:
        return "Check the language pack."
    if "order" in symptom or "layout" in symptom:
        return "Try another PSM mode."
    if "missing" in symptom or "broken" in symptom:
        return "Improve contrast or resolution."
    if "skew" in symptom or "tilted" in symptom:
        return "Deskew the image first."
    return "Inspect the image quality and OCR config."

symptoms = [
    "Special characters are wrong",
    "The text order is strange",
    "Many words are missing",
    "The page is tilted"
]

for symptom in symptoms:
    print(symptom, "=>", explain_ocr_failure(symptom))

## Trick questions

1. Is `pytesseract` the OCR engine?

<details>
<summary>Answer</summary>

No. `pytesseract` is a Python wrapper. The Tesseract app is the OCR engine.

</details>

2. What does `lang="eng+deu"` mean?

<details>
<summary>Answer</summary>

It tells Tesseract to use both English and German language packs.

</details>

3. When should you try `--psm 7`?

<details>
<summary>Answer</summary>

When the image contains one single line of text.

</details>

4. Why are bounding boxes useful?

<details>
<summary>Answer</summary>

They show where each word appears in the image. This helps with layout-aware document extraction.

</details>

5. Should you trust OCR text without checks?

<details>
<summary>Answer</summary>

No. OCR can make mistakes. Clean, validate, and check important fields.

</details>

## Exercises

Fill in each `___`. Run the cell to check your answer.

In [ ]:
# Exercise 1
# Choose the English language code for Tesseract.

lang_code = ___

assert lang_code == "eng"
print("Exercise 1 passed.")

In [ ]:
# Exercise 2
# Build a config string for single block OCR with default engine.

config = ___

assert "--psm 6" in config
assert "--oem 3" in config
print("Exercise 2 passed.")

In [ ]:
# Exercise 3
# Run OCR on the receipt image with the helper function.

text = ___

assert isinstance(text, str)
assert "Total" in text
print("Exercise 3 passed.")

In [ ]:
# Exercise 4
# Clean messy OCR text.

messy = "Hello     OCR\n\nWorld\x0c"
clean = ___

assert clean == "Hello OCR\nWorld"
print("Exercise 4 passed.")

In [ ]:
# Exercise 5
# Get OCR bounding-box data.

box_data = ___

assert isinstance(box_data, pd.DataFrame)
assert {"text", "conf", "left", "top", "width", "height"}.issubset(box_data.columns)
print("Exercise 5 passed.")

In [ ]:
# Exercise 6
# Filter high-confidence OCR words.

high_conf = ___

assert (high_conf["conf"] >= 90).all()
print("Exercise 6 passed.")

In [ ]:
# Exercise 7
# Extract receipt fields from OCR text.

fields = ___

assert fields["date"] == "2026-04-24"
assert abs(fields["total_eur"] - 12.80) < 0.01
print("Exercise 7 passed.")

## Solutions

<details>
<summary>Exercise 1 solution</summary>

```python
lang_code = "eng"
```

</details>

<details>
<summary>Exercise 2 solution</summary>

```python
config = "--psm 6 --oem 3"
```

</details>

<details>
<summary>Exercise 3 solution</summary>

```python
text = run_ocr(receipt_image, lang="eng", config="--psm 6 --oem 3", image_name="receipt")
```

</details>

<details>
<summary>Exercise 4 solution</summary>

```python
clean = clean_ocr_text(messy)
```

</details>

<details>
<summary>Exercise 5 solution</summary>

```python
box_data = get_ocr_data(receipt_image)
```

</details>

<details>
<summary>Exercise 6 solution</summary>

```python
high_conf = ocr_data[ocr_data["conf"] >= 90]
```

</details>

<details>
<summary>Exercise 7 solution</summary>

```python
fields = extract_receipt_fields(clean_receipt_text)
```

</details>

## Cumulative review exercises

These mix topics from Days 11 to 20. Fill in `___` and run each cell.

In [ ]:
# Review 1: Text preprocessing
# Lowercase and split text into tokens.

sentence = "Campaign Text Needs Cleaning"
tokens = ___

assert tokens == ["campaign", "text", "needs", "cleaning"]
print("Review 1 passed.")

In [ ]:
# Review 2: Transformers
# Complete the attention sentence.

attention_word = ___

assert attention_word == "tokens"
print("Review 2 passed.")

In [ ]:
# Review 3: Hugging Face
# Fill the quick helper for pretrained tasks.

hf_helper = ___

assert hf_helper == "pipeline"
print("Review 3 passed.")

In [ ]:
# Review 4: Fine-tuning BERT
# Choose the common Hugging Face class used to train models.

trainer_class = ___

assert trainer_class == "Trainer"
print("Review 4 passed.")

In [ ]:
# Review 5: Complaint classification
# Create a simple label list.

labels = ___

assert isinstance(labels, list)
assert len(labels) >= 3
print("Review 5 passed.")

In [ ]:
# Review 6: OpenAI API
# Fill the standard chat roles.

roles = ___

assert roles == ["system", "user", "assistant"]
print("Review 6 passed.")

In [ ]:
# Review 7: Ollama
# Fill the default local generate endpoint.

ollama_generate_url = ___

assert ollama_generate_url == "http://localhost:11434/api/generate"
print("Review 7 passed.")

In [ ]:
# Review 8: Prompt engineering
# Choose the prompting style that uses examples.

prompting_style = ___

assert prompting_style.lower() == "few-shot"
print("Review 8 passed.")

In [ ]:
# Review 9: Structured output
# Parse JSON text into a Python dict.

json_text = '{"campaign": "Demo", "clicks": 100}'
parsed = ___

assert parsed["clicks"] == 100
print("Review 9 passed.")

In [ ]:
# Review 10: Information extraction
# Calculate conversion rate from structured fields.

record = {"clicks": 1000, "conversions": 80}
conversion_rate = ___

assert abs(conversion_rate - 0.08) < 1e-9
print("Review 10 passed.")

## Cumulative review solutions

<details>
<summary>Show solutions</summary>

```python
# Review 1
tokens = sentence.lower().split()

# Review 2
attention_word = "tokens"

# Review 3
hf_helper = "pipeline"

# Review 4
trainer_class = "Trainer"

# Review 5
labels = ["billing", "delivery", "technical"]

# Review 6
roles = ["system", "user", "assistant"]

# Review 7
ollama_generate_url = "http://localhost:11434/api/generate"

# Review 8
prompting_style = "few-shot"

# Review 9
parsed = json.loads(json_text)

# Review 10
conversion_rate = record["conversions"] / record["clicks"]
```

</details>

In [ ]:
cheat_sheet = '''
DAY 21 CHEAT SHEET: TESSERACT BASICS

Core idea:
- OCR reads text from images.
- Tesseract is the OCR engine.
- pytesseract is the Python wrapper.

Install:
- macOS: brew install tesseract
- Python: pip install pytesseract pillow

Language packs:
- English: eng
- German: deu
- Turkish: tur
- Mixed: eng+deu or eng+tur

Common config:
- --psm 6: one block of text
- --psm 7: one line of text
- --psm 8: one word
- --oem 3: default OCR engine

Useful functions:
- pytesseract.image_to_string(image)
- pytesseract.image_to_data(image)
- clean OCR text before extraction
- validate important fields after extraction
'''

print(cheat_sheet)

## Next up: Day 22 — EasyOCRAndComparison

You will compare EasyOCR and Tesseract, including setup, strengths, weaknesses, and result quality.